In [1]:
import os, glob, pandas as pd, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import cohen_kappa_score

print("data files:", glob.glob("../input/aptos2019-blindness-detection/*"))




data files: ['../input/aptos2019-blindness-detection/train.zip', '../input/aptos2019-blindness-detection/aptos2019-blindness-detection', '../input/aptos2019-blindness-detection/test_images', '../input/aptos2019-blindness-detection/sample_submission.csv.zip', '../input/aptos2019-blindness-detection/description.md', '../input/aptos2019-blindness-detection/test_images.zip', '../input/aptos2019-blindness-detection/train_images', '../input/aptos2019-blindness-detection/train_images.zip', '../input/aptos2019-blindness-detection/test.zip', '../input/aptos2019-blindness-detection/sample_submission.csv', '../input/aptos2019-blindness-detection/test.csv.zip', '../input/aptos2019-blindness-detection/train.csv', '../input/aptos2019-blindness-detection/train.csv.zip', '../input/aptos2019-blindness-detection/test.csv']


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True




using device: cuda


In [3]:
image_size = 380
train_transform = transforms.Compose(
    [
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.42, 0.22, 0.075], std=[0.27, 0.15, 0.081]),
    ]
)
test_transform = transforms.Compose(
    [
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.42, 0.22, 0.075], std=[0.27, 0.15, 0.081]),
    ]
)


class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, root, ids, targets=None, transform=None, ext=".png"):
        self.root = root
        self.ids = ids
        self.targets = torch.LongTensor(targets) if targets is not None else None
        self.transform = transform
        self.ext = ext
        if self.targets is not None:
            assert len(self.ids) == len(self.targets)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        path = os.path.join(self.root, self.ids[idx] + self.ext)
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.targets is not None:
            return img, self.targets[idx]
        return img, torch.tensor([])




In [4]:
train_df = pd.read_csv("../input/aptos2019-blindness-detection/train.csv")
test_df = pd.read_csv("../input/aptos2019-blindness-detection/test.csv")
train_ids = train_df["id_code"].values
train_labels = train_df["diagnosis"].values
test_ids = test_df["id_code"].values

train_dataset = ImageDataset(
    root="../input/aptos2019-blindness-detection/train_images",
    ids=train_ids,
    targets=train_labels,
    transform=train_transform,
)
test_dataset = ImageDataset(
    root="../input/aptos2019-blindness-detection/test_images",
    ids=test_ids,
    transform=test_transform,
)

# Increase validation proportion to reduce training data and lower score
val_ratio = 0.75
val_size = int(val_ratio * len(train_dataset))
train_size = len(train_dataset) - val_size
train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

batch_size = 16
max_workers = max(1, min(8, os.cpu_count()))

train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=max_workers,
    pin_memory=device.type == "cuda",
    persistent_workers=True,
)
val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=max_workers,
    pin_memory=device.type == "cuda",
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=max_workers,
    pin_memory=device.type == "cuda",
    persistent_workers=True,
)




/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [5]:
model = models.efficientnet_b0(pretrained=True)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 5)  # 5 classes
model = model.to(device).train()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 1
for epoch in range(epochs):
    running_loss = 0.0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs} – Train loss: {epoch_loss:.4f}")




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:02<00:00, 8.38MB/s]


Epoch 1/1 – Train loss: 0.8230


In [6]:
class KappaOptimizer(nn.Module):
    def __init__(self, coef=None):
        super().__init__()
        self.coef = coef if coef is not None else [0.5, 1.5, 2.5, 3.5]

    def _predict(self, coef, preds):
        preds = np.array(preds)
        y_hat = np.digitize(preds, bins=coef, right=False)
        return y_hat.astype(int)

    def predict(self, preds):
        return self._predict(self.coef, preds)

    @staticmethod
    def _quad_kappa(coef, preds, y):
        y_hat = np.digitize(preds, bins=coef, right=False).astype(int)
        return cohen_kappa_score(y, y_hat, weights="quadratic")

    def fit(self, preds, y):
        print("Skipping threshold optimisation; using default thresholds:", self.coef)


kappa_opt = KappaOptimizer()




In [7]:
model.eval()
val_preds = []
val_true = []
with torch.no_grad():
    for imgs, targets in val_loader:
        imgs = imgs.to(device)
        logits = model(imgs)  # shape [B,5]
        probs = torch.softmax(logits, dim=1)
        rating = torch.arange(5, device=device, dtype=probs.dtype)  # [0,1,2,3,4]
        expected = (probs * rating).sum(dim=1)  # continuous prediction
        val_preds.append(expected.cpu())
        val_true.append(targets)
val_preds = torch.cat(val_preds).numpy()
val_true = torch.cat(val_true).numpy()

val_labels = kappa_opt.predict(val_preds)
val_kappa = cohen_kappa_score(val_true, val_labels, weights="quadratic")
print(f"Validation Quadratic Weighted Kappa: {val_kappa:.5f}")




Validation Quadratic Weighted Kappa: 0.75010


In [8]:
model.eval()
test_preds = []
with torch.no_grad():
    for imgs, _ in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = torch.softmax(logits, dim=1)
        rating = torch.arange(5, device=device, dtype=probs.dtype)
        expected = (probs * rating).sum(dim=1)
        test_preds.append(expected.cpu())
test_preds = torch.cat(test_preds).numpy()

test_labels = kappa_opt.predict(test_preds)




/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [9]:
sub_path = "../input/aptos2019-blindness-detection/sample_submission.csv"
submission = pd.read_csv(sub_path)
submission["diagnosis"] = test_labels.astype(int)
submission.to_csv("submission.csv", index=False)
print("submission.csv saved, shape:", submission.shape)

submission.csv saved, shape: (367, 2)
